# Pelajaran 10 - Agen AI dalam Produksi

Dalam pelajaran ini Anda akan mempelajari **pola produksi** untuk agen AI menggunakan Microsoft Agent Framework (Python). Kami membahas:

- **Observabilitas** — menambahkan pengukuran waktu dan pencatatan ke interaksi agen
- **Evaluasi** — menggunakan agen penilai untuk memberi skor kualitas respons
- **Manajemen biaya** — strategi untuk optimisasi token dan pemilihan model

Skenarionya adalah sebuah **agen perjalanan** yang membantu pengguna merencanakan perjalanan, dengan pemantauan dan evaluasi di atasnya.


## Persiapan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Pertimbangan Produksi

Memindahkan agen AI dari prototipe ke produksi membutuhkan perhatian yang cermat terhadap tiga pilar:

1. **Observabilitas** — Anda memerlukan visibilitas terhadap apa yang dilakukan agen, berapa lama waktu yang dibutuhkan, dan alat apa yang dipanggilnya. Tanpa penelusuran dan pencatatan, penyelesaian masalah produksi hampir mustahil.

2. **Evaluasi** — Pemeriksaan kualitas otomatis memastikan jawaban agen tetap akurat, lengkap, dan membantu dari waktu ke waktu. Agen penilai dapat memberi skor jawaban terhadap kriteria yang ditetapkan.

3. **Manajemen Biaya** — Penggunaan token secara langsung memengaruhi biaya. Strategi seperti optimasi prompt, pemilihan model, dan caching membantu menjaga pengeluaran tetap terkendali tanpa mengorbankan kualitas.


## Membangun Agen yang Dapat Diamati

Kami mendefinisikan alat perjalanan dan membungkus pemanggilan agen dengan pengukuran waktu sehingga kami dapat memantau latensi. Di lingkungan produksi Anda akan mengintegrasikannya dengan OpenTelemetry atau backend pelacakan serupa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Pola Evaluasi

Sebuah pola produksi umum adalah menggunakan agen kedua sebagai **penilai**. Penilai memberi skor tanggapan agen utama berdasarkan kriteria yang telah ditentukan seperti kelengkapan, akurasi, dan kegunaan.

Hal ini memungkinkan:
- Gerbang kualitas otomatis sebelum tanggapan sampai ke pengguna
- Deteksi regresi ketika prompt atau model berubah
- Pemantauan berkelanjutan terhadap kinerja agen dari waktu ke waktu


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategi Pengelolaan Biaya

Mengendalikan biaya sangat penting untuk agen AI produksi. Berikut strategi utama:

| Strategi | Deskripsi |
|---|---|
| **Optimisasi prompt** | Jaga instruksi sistem tetap ringkas. Hapus konteks yang berlebihan untuk mengurangi token input. |
| **Pemilihan model** | Gunakan model yang lebih kecil dan lebih murah (mis. GPT-4o-mini) untuk tugas sederhana seperti klasifikasi atau ekstraksi, dan simpan model yang lebih besar untuk penalaran yang kompleks. |
| **Caching** | Simpan hasil alat dan kueri yang sering untuk menghindari panggilan API yang berulang. |
| **Batas token** | Tetapkan batas `max_tokens` untuk mencegah respons yang tak terduga panjangnya. |
| **Pengelompokan** | Kelompokkan beberapa kueri pengguna menjadi satu panggilan API bila memungkinkan. |

Dalam praktiknya, pendekatan bertingkat bekerja dengan baik: arahkan permintaan yang sederhana ke model yang cepat dan murah, dan tingkatkan hanya kueri yang kompleks ke model yang lebih mampu.


## Ringkasan

Dalam pelajaran ini Anda belajar cara untuk:

1. **Menambahkan observabilitas** pada interaksi agen dengan pengukuran waktu dan pencatatan, meletakkan dasar untuk penelusuran dan pemantauan.
2. **Mengevaluasi respons agen** secara otomatis menggunakan agen evaluator yang memberi skor kelengkapan, akurasi, dan kegunaan.
3. **Mengelola biaya** melalui optimasi prompt, pemilihan model, caching, dan anggaran token.

Pola-pola produksi ini membantu memastikan agen AI Anda dapat diandalkan, terukur, dan hemat biaya dalam skala besar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya mencapai ketepatan, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang berwenang. Untuk informasi yang bersifat kritis, disarankan menggunakan terjemahan profesional oleh penerjemah manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
